# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors including MSI-H Status and Anatomical Distribution Exploration with `mlcroissant`

This notebook demonstrates how to load and explore the FAIR² colorectal cancer survivor dataset using the [`mlcroissant`](https://mlcommons.github.io/croissant-python/) library following best practices for the Croissant schema.

### Dataset Source

- FAIR² dataset (Clinical, molecular, and pathological data for 77 colorectal cancer survivors)
- Source (Croissant schema): [https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json](https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json)

In [ ]:
# Ensure mlcroissant library is installed
!pip install mlcroissant

## 1. Data Loading

Load metadata and available records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import pprint

# Define the Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json'

# Load the dataset
dataset = mlc.Dataset(croissant_url)

# Get metadata object
metadata = dataset.metadata

print('--- DATASET METADATA ---')
print('Dataset title:', metadata.name)
print('Identifier:', metadata.identifier)
print('Version:', metadata.version)
print('Published:', metadata.datePublished)
print('License:', metadata.license)
print('Description:', metadata.description)
print('Personal Sensitive Information:', getattr(metadata, 'personalSensitiveInformation', 'Not listed'))


## 2. Data Overview

Review available **record sets**, **fields**, and their `@id` identifiers.

This step is crucial for understanding the structure of the Croissant schema and mapping which data tables (record sets) are available, along with their field `@id`s.

In [ ]:
# List all available record sets and their fields with @id
record_set_infos = []
print('--- Available Record Sets ---')
for record_set in dataset.record_sets():
    print(f"Record set @id: {record_set.id}")
    print(f"  Name: {getattr(record_set, 'name', '[No name]')}")
    print(f"  Description: {getattr(record_set, 'description', '[No description]')}")
    print(f"  Fields:")
    for field in record_set.fields:
        print(f"    - Field @id: {field.id} (name: {field.name}, dataType: {getattr(field, 'dataType', 'N/A')})")
    record_set_infos.append({'@id': record_set.id, 'fields': [field.id for field in record_set.fields]})
    print('')

# Save available record set ids and field ids for later use
record_set_ids = [info['@id'] for info in record_set_infos]

## 3. Data Extraction

Load records from record sets into pandas DataFrames for analysis. 

**Note**: All data will be referenced by their schema `@id`s.

In [ ]:
# Extract data from each record set into pandas DataFrames

dataframes = {}
for rset_id in record_set_ids:
    records = list(dataset.records(record_set=rset_id))
    if len(records) > 0:
        df = pd.DataFrame(records)
        dataframes[rset_id] = df
        print(f"Loaded record set '@id': {rset_id}")
        print(f"  Shape: {df.shape}")
        print(f"  Columns (@id): {list(df.columns)}\n")
    else:
        print(f"Record set '@id': {rset_id} contains no records. Skipped.\n")

# Pick the first non-empty DataFrame as example for further EDA/analysis
main_record_set_id = next((k for k, v in dataframes.items() if len(v)>0), None)
assert main_record_set_id is not None, "No non-empty record sets found."
print(f"Selected record set for analysis: {main_record_set_id}")
main_df = dataframes[main_record_set_id]
print(f"\nPreview of '{main_record_set_id}' records:")
display(main_df.head())

## 4. Exploratory Data Analysis (EDA)

Let's perform some standard EDA steps using only columns referenced by their `@id`. 

- We select a **numeric field** for demonstration.
- Filter records, normalize, and group by a categorical variable (if present).

In [ ]:
# Identify a numeric field to analyze by checking dtype
numeric_field_id = None
for col in main_df.columns:
    # Try to infer numeric columns by dtype or naming
    if pd.api.types.is_numeric_dtype(main_df[col]):
        numeric_field_id = col
        break

if numeric_field_id is None:
    # As fallback, try to convert any columns to numeric
    for col in main_df.columns:
        try:
            _ = pd.to_numeric(main_df[col].dropna().iloc[:10])
            numeric_field_id = col
            break
        except Exception:
            continue

if numeric_field_id is not None:
    # Try to coerce the entire column to numeric
    main_df[numeric_field_id] = pd.to_numeric(main_df[numeric_field_id], errors='coerce')
    threshold = main_df[numeric_field_id].mean() if main_df[numeric_field_id].notnull().any() else 0
    filtered_df = main_df[main_df[numeric_field_id] > threshold]
    print(f"Filtered records with {numeric_field_id} > {threshold:.2f}:")
    display(filtered_df.head())
    
    filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
    print(f"Normalized {numeric_field_id} for filtered records:")
    display(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())
    
    # Find a likely grouping/categorical column
    group_field_id = None
    for col in main_df.columns:
        if main_df[col].dtype == object and col != numeric_field_id:
            nunique = main_df[col].nunique()
            if 1 < nunique < len(main_df) // 2:
                group_field_id = col
                break
    if group_field_id is not None:
        grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
        print(f"Grouped mean of {numeric_field_id} by {group_field_id}:")
        display(grouped_df.head())
else:
    print('No numeric field could be detected in the main record set for EDA demo.')

## 5. Visualization

Plot the distribution of the selected numeric field, and (if grouped) compare group means.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if numeric_field_id is not None:
    plt.figure(figsize=(6,4))
    sns.histplot(main_df[numeric_field_id].dropna(), kde=True)
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.ylabel('Count')
    plt.tight_layout()
    plt.show()
    
    if 'grouped_df' in locals() and group_field_id is not None:
        plt.figure(figsize=(8,4))
        sns.barplot(x=group_field_id, y=numeric_field_id, data=grouped_df)
        plt.title(f"Mean {numeric_field_id} by {group_field_id}")
        plt.tight_layout()
        plt.show()


## 6. Conclusion

- This notebook demonstrated how to load and explore a FAIR² clinical dataset defined by a Croissant schema using the `mlcroissant` library.
- We programmatically listed available record sets and fields by their `@id`s, loaded and inspected the main table, performed simple filtering and normalizing, and visualized a variable distribution.
- All operations referenced schema objects by their `@id` as recommended for robust and reproducible analysis.

**Next steps:** deeper statistical analysis, join with external knowledge sources, or use fields and record sets explicitly by their `@id` for machine learning or downstream clinical modeling.